In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


file = "TS hour raw value 2024 L1&2.xlsx"
df = pd.read_excel(file)

df.head()


# ============================================================
# SABİTLER
# ============================================================

CPA = 1.3104
CPF_TFUM = 1.087
CPF_TREF = 1.225
NCV_GAS = 33500.0
TREF = 15.0
VWA = 5.56
VO2_MCR = 0.0764
COLD_AIR_INJECTION = 5200.0 / 3600.0
H_EA = 551.0
H_WH = 3216.0
LOSS = 0.04


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    """Nm3/h -> Nm3/s"""
    return x / 3600.0


def th_to_kgs(x):
    """ton/h -> kg/s"""
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    """
    Yüzde değerlerini fraction'a çevirir.

    Örnek:
    8.5 -> 0.085
    0.085 -> 0.085
    """
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    """Negatif değerleri sıfıra çeker."""
    return np.maximum(x, 0)


# ============================================================
# KOLON EŞLEŞTİRME
# ============================================================

def get_cols(prefix):

    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2": f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    # Waste Flow kolon isimleri L1 ve L2 için farklı
    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Bilinmeyen line prefix: {prefix}")

    return base


# ============================================================
# KOLON KONTROLÜ
# ============================================================

def validate_columns(df, cols, line_name):

    missing = [
        column
        for column in cols.values()
        if column not in df.columns
    ]

    if missing:
        print(f"\n{line_name} için eksik kolonlar:")

        for column in missing:
            print(f"  - {column}")

        raise KeyError(
            f"{line_name} hesabı yapılamadı. "
            f"{len(missing)} adet gerekli kolon Excel dosyasında bulunamadı."
        )


# ============================================================
# ANA HESAPLAMA
# ============================================================

def compute_line(df, prefix):

    cols = get_cols(prefix)

    line_name = "L1" if prefix == "1_" else "L2"

    # Önce kolonları kontrol et
    validate_columns(df, cols, line_name)

    # Orijinal dataframe'i değiştirmemek için copy
    data = df.copy()

    # Kullanılacak kolonları numeric yap
    for column in cols.values():
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        )

    # --------------------------------------------------------
    # FLOW CONVERSIONS
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(data[cols["W_AP"]])
    W_AS = nm3h_to_nm3s(data[cols["W_AS"]])

    W_GR = nm3h_to_nm3s(data[cols["W_GR"]])
    W_GL = nm3h_to_nm3s(data[cols["W_GL"]])

    W_WH = th_to_kgs(data[cols["W_WH"]])
    W_OM = th_to_kgs(data[cols["W_OM"]])

    # --------------------------------------------------------
    # TEMPERATURES
    # --------------------------------------------------------

    T_AP = data[cols["T_AP"]]
    T_AS = data[cols["T_AS"]]
    T_FUM = data[cols["TFUM"]]

    # --------------------------------------------------------
    # FLUE GAS ANALYSIS
    # --------------------------------------------------------

    VO2 = pct_to_frac(data[cols["VO2"]])
    VH2O = pct_to_frac(data[cols["VH2O"]])

    # --------------------------------------------------------
    # FLUE GAS FLOW
    # --------------------------------------------------------

    denominator = 1 - VH2O

    # VH2O = 100% olması gibi matematiksel hataları önle
    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    # --------------------------------------------------------
    # FLUE GAS HEAT
    # --------------------------------------------------------

    PFUM = WFUM * (
        CPF_TFUM * T_FUM
        - CPF_TREF * TREF
    )

    # --------------------------------------------------------
    # AIR HEAT
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        )
        * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # WATER / STEAM ENERGY
    # --------------------------------------------------------

    QNR = W_WH * (
        H_WH - H_EA
    )

    # --------------------------------------------------------
    # GAS ENERGY
    # --------------------------------------------------------

    QGAS = NCV_GAS * (
        W_GR + W_GL
    )

    # --------------------------------------------------------
    # WASTE ENERGY
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV = QOM / W_OM.replace(
        0,
        np.nan
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    result = pd.DataFrame({
        "NCV": NCV,
        "QOM": QOM
    })

    return result


# ============================================================
# MAIN
# ============================================================

def main():

    # Bu Python dosyasının bulunduğu klasör
    BASE_DIR = Path.cwd()

    # Excel input
    input_file = (
        BASE_DIR
        / "data"
        / "TS hour raw value 2024 L1&2.xlsx"
    )

    # Output klasörü
    output_dir = BASE_DIR / "output"

    # Yoksa oluştur
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Output Excel
    output_file = (
        output_dir
        / "NCV_2024_hourly_L1_L2.xlsx"
    )

    # --------------------------------------------------------
    # INPUT FILE CONTROL
    # --------------------------------------------------------

    if not input_file.exists():

        raise FileNotFoundError(
            "\nInput Excel dosyası bulunamadı.\n"
            f"Beklenen konum:\n{input_file}\n\n"
            "Excel dosyasını repository içerisindeki "
            "'data' klasörüne koy."
        )

    print("=" * 60)
    print("NCV CALCULATION")
    print("=" * 60)

    print(f"\nInput:\n{input_file}")

    # --------------------------------------------------------
    # READ EXCEL
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        input_file,
        engine="openpyxl"
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satır x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # CALCULATIONS
    # --------------------------------------------------------

    print("\nL1 hesaplanıyor...")
    L1 = compute_line(df, "1_")

    print("L2 hesaplanıyor...")
    L2 = compute_line(df, "2_")

    # --------------------------------------------------------
    # SAVE EXCEL
    # --------------------------------------------------------

    print("\nSonuç Excel'e yazılıyor...")

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        L1.to_excel(
            writer,
            index=False,
            sheet_name="L1"
        )

        L2.to_excel(
            writer,
            index=False,
            sheet_name="L2"
        )

    print("\n" + "=" * 60)
    print("EXCEL OLUŞTURULDU")
    print("=" * 60)

    print(output_file)

    # --------------------------------------------------------
    # QUICK SUMMARY
    # --------------------------------------------------------

    print("\nL1 NCV Summary:")
    print(L1["NCV"].describe())

    print("\nL2 NCV Summary:")
    print(L2["NCV"].describe())


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


file = "TS hour raw value 2024 L1&2.xlsx"
df = pd.read_excel(file)

df.head()


# ============================================================
# SABİTLER
# ============================================================

CPA = 1.3104
CPF_TFUM = 1.087
CPF_TREF = 1.225
NCV_GAS = 33500.0
TREF = 15.0
VWA = 5.56
VO2_MCR = 0.0764
COLD_AIR_INJECTION = 5200.0 / 3600.0
H_EA = 551.0
H_WH = 3216.0
LOSS = 0.04


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    """Nm3/h -> Nm3/s"""
    return x / 3600.0


def th_to_kgs(x):
    """ton/h -> kg/s"""
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    """
    Yüzde değerlerini fraction'a çevirir.

    Örnek:
    8.5 -> 0.085
    0.085 -> 0.085
    """
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    """Negatif değerleri sıfıra çeker."""
    return np.maximum(x, 0)


# ============================================================
# KOLON EŞLEŞTİRME
# ============================================================

def get_cols(prefix):

    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2": f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    # Waste Flow kolon isimleri L1 ve L2 için farklı
    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Bilinmeyen line prefix: {prefix}")

    return base


# ============================================================
# KOLON KONTROLÜ
# ============================================================

def validate_columns(df, cols, line_name):

    missing = [
        column
        for column in cols.values()
        if column not in df.columns
    ]

    if missing:
        print(f"\n{line_name} için eksik kolonlar:")

        for column in missing:
            print(f"  - {column}")

        raise KeyError(
            f"{line_name} hesabı yapılamadı. "
            f"{len(missing)} adet gerekli kolon Excel dosyasında bulunamadı."
        )


# ============================================================
# ANA HESAPLAMA
# ============================================================

def compute_line(df, prefix):

    cols = get_cols(prefix)

    line_name = "L1" if prefix == "1_" else "L2"

    # Önce kolonları kontrol et
    validate_columns(df, cols, line_name)

    # Orijinal dataframe'i değiştirmemek için copy
    data = df.copy()

    # Kullanılacak kolonları numeric yap
    for column in cols.values():
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        )

    # --------------------------------------------------------
    # FLOW CONVERSIONS
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(data[cols["W_AP"]])
    W_AS = nm3h_to_nm3s(data[cols["W_AS"]])

    W_GR = nm3h_to_nm3s(data[cols["W_GR"]])
    W_GL = nm3h_to_nm3s(data[cols["W_GL"]])

    W_WH = th_to_kgs(data[cols["W_WH"]])
    W_OM = th_to_kgs(data[cols["W_OM"]])

    # --------------------------------------------------------
    # TEMPERATURES
    # --------------------------------------------------------

    T_AP = data[cols["T_AP"]]
    T_AS = data[cols["T_AS"]]
    T_FUM = data[cols["TFUM"]]

    # --------------------------------------------------------
    # FLUE GAS ANALYSIS
    # --------------------------------------------------------

    VO2 = pct_to_frac(data[cols["VO2"]])
    VH2O = pct_to_frac(data[cols["VH2O"]])

    # --------------------------------------------------------
    # FLUE GAS FLOW
    # --------------------------------------------------------

    denominator = 1 - VH2O

    # VH2O = 100% olması gibi matematiksel hataları önle
    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    # --------------------------------------------------------
    # FLUE GAS HEAT
    # --------------------------------------------------------

    PFUM = WFUM * (
        CPF_TFUM * T_FUM
        - CPF_TREF * TREF
    )

    # --------------------------------------------------------
    # AIR HEAT
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        )
        * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # WATER / STEAM ENERGY
    # --------------------------------------------------------

    QNR = W_WH * (
        H_WH - H_EA
    )

    # --------------------------------------------------------
    # GAS ENERGY
    # --------------------------------------------------------

    QGAS = NCV_GAS * (
        W_GR + W_GL
    )

    # --------------------------------------------------------
    # WASTE ENERGY
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV = QOM / W_OM.replace(
        0,
        np.nan
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    result = pd.DataFrame({
        "NCV": NCV,
        "QOM": QOM
    })

    return result


# ============================================================
# MAIN
# ============================================================

def main():

    # Bu Python dosyasının bulunduğu klasör
    BASE_DIR = Path.cwd()

    # Excel input
    input_file = (
        BASE_DIR
        / "data"
        / "TS hour raw value 2024 L1&2.xlsx"
    )

    # Output klasörü
    output_dir = BASE_DIR / "output"

    # Yoksa oluştur
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Output Excel
    output_file = (
        output_dir
        / "NCV_2024_hourly_L1_L2.xlsx"
    )

    # --------------------------------------------------------
    # INPUT FILE CONTROL
    # --------------------------------------------------------

    if not input_file.exists():

        raise FileNotFoundError(
            "\nInput Excel dosyası bulunamadı.\n"
            f"Beklenen konum:\n{input_file}\n\n"
            "Excel dosyasını repository içerisindeki "
            "'data' klasörüne koy."
        )

    print("=" * 60)
    print("NCV CALCULATION")
    print("=" * 60)

    print(f"\nInput:\n{input_file}")

    # --------------------------------------------------------
    # READ EXCEL
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        input_file,
        engine="openpyxl"
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satır x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # CALCULATIONS
    # --------------------------------------------------------

    print("\nL1 hesaplanıyor...")
    L1 = compute_line(df, "1_")

    print("L2 hesaplanıyor...")
    L2 = compute_line(df, "2_")

    # --------------------------------------------------------
    # SAVE EXCEL
    # --------------------------------------------------------

    print("\nSonuç Excel'e yazılıyor...")

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        L1.to_excel(
            writer,
            index=False,
            sheet_name="L1"
        )

        L2.to_excel(
            writer,
            index=False,
            sheet_name="L2"
        )

    print("\n" + "=" * 60)
    print("EXCEL OLUŞTURULDU")
    print("=" * 60)

    print(output_file)

    # --------------------------------------------------------
    # QUICK SUMMARY
    # --------------------------------------------------------

    print("\nL1 NCV Summary:")
    print(L1["NCV"].describe())

    print("\nL2 NCV Summary:")
    print(L2["NCV"].describe())


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
#  DOSYA YOLLARI - GITHUB CODESPACES / JUPYTER
# ============================================================

BASE_DIR = Path.cwd()

INPUT_FILE = (
    BASE_DIR
    / "data"
    / "TS hour raw value 2024 L1&2.xlsx"
)

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "NCV_2024_hourly_L1_L2.xlsx"
)


# ============================================================
#  SABITLER
# ============================================================

NCV_GAS = 33500.0      # kJ/Nm3
TREF = 15.0            # C
VO2_MCR = 0.0764       # -
COLD_AIR_INJECTION = 5200.0 / 3600.0   # Nm3/s
H_EA = 551.0           # kJ/kg
H_WH = 3216.0          # kJ/kg
LOSS = 0.04            # %4 kayip

BASE_CONST = {
    "CPA": 1.3104,
    "CPF_TFUM": 1.3833,
    "CPF_TREF": 1.3574,
    "VWA": 3.94,
}

# NCV alt limiti (kJ/kg)
NCV_MIN = 3500.0

# 4 saatlik ortalama waste flow alt limiti (t/h)
MIN_WASTE_FLOW_TPH = 15.0


# ============================================================
#  YARDIMCI FONKSIYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    return x / 3600.0


def th_to_kgs(x):
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    return np.maximum(x, 0.0)


def get_cols(prefix: str) -> dict:
    """
    Verilen prefix icin ('1_' veya '2_') KKS tag isimlerini dondurur.
    """
    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2":  f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Gecersiz prefix: {prefix}")

    return base


def validate_columns(df: pd.DataFrame, cols: dict, label: str):
    missing = [c for c in cols.values() if c not in df.columns]

    if missing:
        print(f"\n{label} icin eksik kolonlar:")
        for c in missing:
            print(" -", c)

        raise KeyError(
            f"{label} hesabi yapilamadi. "
            f"{len(missing)} gerekli kolon Excel dosyasinda yok."
        )


# ============================================================
#  TEK HAT HESABI
# ============================================================

def compute_line(df: pd.DataFrame, prefix: str, label: str) -> pd.DataFrame:
    """
    Tek bir hat (L1 veya L2) icin NCV, QOM ve diger isi terimlerini hesaplar.

    W_OM hesaplamalarda 4 saatlik hareketli ortalama ile kullanilir.
    Eski kod mantigini korumak icin min_periods=1 kullanilir.

    NCV < 3500 kJ/kg olan saatlerde NCV NaN yapilir.
    """

    cols = get_cols(prefix)
    validate_columns(df, cols, label)

    def col(cname):
        return pd.to_numeric(df[cname], errors="coerce")

    # --------------------------------------------------------
    # AKISLAR
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(col(cols["W_AP"]))
    W_AS = nm3h_to_nm3s(col(cols["W_AS"]))

    W_GR = nm3h_to_nm3s(col(cols["W_GR"]))
    W_GL = nm3h_to_nm3s(col(cols["W_GL"]))

    W_WH_tph = col(cols["W_WH"])
    W_WH = th_to_kgs(W_WH_tph)

    # Ham waste flow t/h
    W_OM_tph_raw = col(cols["W_OM"])

    # 4 saatlik hareketli ortalama
    W_OM_tph_4h = (
        W_OM_tph_raw
        .rolling(window=4, min_periods=1)
        .mean()
    )

    # 4 saatlik ortalama kg/s
    W_OM = th_to_kgs(W_OM_tph_4h)

    # --------------------------------------------------------
    # SICAKLIKLAR
    # --------------------------------------------------------

    T_AP = col(cols["T_AP"])
    T_AS = col(cols["T_AS"])
    T_FUM = col(cols["TFUM"])

    # --------------------------------------------------------
    # GAZ ANALIZI
    # --------------------------------------------------------

    VO2 = pct_to_frac(col(cols["VO2"]))
    VH2O = pct_to_frac(col(cols["VH2O"]))

    # --------------------------------------------------------
    # KATSAYILAR
    # --------------------------------------------------------

    VWA = BASE_CONST["VWA"]
    CPF_TFUM = BASE_CONST["CPF_TFUM"]
    CPF_TREF = BASE_CONST["CPF_TREF"]
    CPA = BASE_CONST["CPA"]

    # --------------------------------------------------------
    # BACAGAZI
    # --------------------------------------------------------

    denominator = 1 - VH2O

    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    PFUM = (
        WFUM
        * (
            CPF_TFUM * T_FUM
            - CPF_TREF * TREF
        )
    )

    # --------------------------------------------------------
    # HAVA ISITMA
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        ) * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # BUHAR / SU
    # --------------------------------------------------------

    QNR = W_WH * (H_WH - H_EA)

    # --------------------------------------------------------
    # DOGAL GAZ
    # --------------------------------------------------------
    #
    # NOT:
    # Orijinal eski kodunda /1000 vardi.
    # Bu dosyada eski hesap mantigini korumak icin aynen birakildi.
    #

    QGAS = NCV_GAS * (W_GR + W_GL) / 1000.0

    # --------------------------------------------------------
    # ATIKTAN GELEN ISI
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV_raw = (
        QOM
        / W_OM.replace(0, np.nan)
    )

    # 3500 kJ/kg altini NaN yap
    NCV = NCV_raw.where(
        NCV_raw >= NCV_MIN,
        np.nan
    )

    # --------------------------------------------------------
    # SONUC
    # --------------------------------------------------------

    return pd.DataFrame({
        "Time": df["Time"],

        f"{label}_NCV": NCV,
        f"{label}_QOM": QOM,
        f"{label}_QGAS": QGAS,
        f"{label}_QNR": QNR,
        f"{label}_PFUM": PFUM,
        f"{label}_QAIR": QAIR,

        # 4 saatlik ortalama waste flow
        f"{label}_W_OM_kg_s": W_OM,

        # Ham saatlik waste flow
        f"{label}_W_OM_tph_raw": W_OM_tph_raw,

        # Bilgi amacli 4 saatlik ortalama t/h
        f"{label}_W_OM_tph_4h": W_OM_tph_4h,
    })


# ============================================================
#  ANA CALISMA
# ============================================================

def main():

    # --------------------------------------------------------
    # 1) INPUT KONTROL
    # --------------------------------------------------------

    print("=" * 70)
    print("NCV CALCULATION")
    print("=" * 70)

    print("\nCalisma klasoru:")
    print(BASE_DIR)

    print("\nInput dosyasi:")
    print(INPUT_FILE)

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            "\nExcel dosyasi bulunamadi.\n"
            f"Beklenen konum:\n{INPUT_FILE}\n\n"
            "Excel dosyasini repository icindeki data klasorune koy."
        )

    # --------------------------------------------------------
    # 2) VERI OKU
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        INPUT_FILE,
        engine="openpyxl"
    )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    # Ilk kolonu Time olarak kullan
    first_col = df.columns[0]

    if first_col != "Time":
        df = df.rename(
            columns={
                first_col: "Time"
            }
        )

    df["Time"] = pd.to_datetime(
        df["Time"],
        errors="coerce"
    )

    df = (
        df
        .sort_values("Time")
        .reset_index(drop=True)
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satir x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # 3) L1 / L2 HESAPLAMA
    # --------------------------------------------------------

    print("\nL1 hesaplaniyor...")
    L1 = compute_line(
        df,
        "1_",
        "L1"
    )

    print("L2 hesaplaniyor...")
    L2 = compute_line(
        df,
        "2_",
        "L2"
    )

    hourly = pd.merge(
        L1,
        L2,
        on="Time",
        how="outer"
    )

    # --------------------------------------------------------
    # 4) WASTE FLOW VALIDITY
    # --------------------------------------------------------

    print("\nDurus/kalkis/trip saatleri filtreleniyor...")

    min_waste_kg_s = th_to_kgs(
        MIN_WASTE_FLOW_TPH
    )

    # 4 saatlik ortalama W_OM >= 15 t/h ise valid
    hourly["valid_L1"] = (
        hourly["L1_W_OM_kg_s"]
        >= min_waste_kg_s
    )

    hourly["valid_L2"] = (
        hourly["L2_W_OM_kg_s"]
        >= min_waste_kg_s
    )

    # Geçersiz saatlerde hesap kolonlarini NaN yap
    # Waste flow bilgi kolonlarini koru

    protected_L1 = [
        "L1_W_OM_kg_s",
        "L1_W_OM_tph_raw",
        "L1_W_OM_tph_4h",
    ]

    protected_L2 = [
        "L2_W_OM_kg_s",
        "L2_W_OM_tph_raw",
        "L2_W_OM_tph_4h",
    ]

    cols_L1 = [
        c
        for c in hourly.columns
        if c.startswith("L1_")
        and c not in protected_L1
    ]

    cols_L2 = [
        c
        for c in hourly.columns
        if c.startswith("L2_")
        and c not in protected_L2
    ]

    hourly.loc[
        ~hourly["valid_L1"],
        cols_L1
    ] = np.nan

    hourly.loc[
        ~hourly["valid_L2"],
        cols_L2
    ] = np.nan

    # --------------------------------------------------------
    # 5) PLANT
    # --------------------------------------------------------
    #
    # Eski kod mantigi:
    # Plant NCV = L1/L2 NCV aritmetik ortalamasi
    #

    hourly["Plant_NCV"] = (
        hourly[
            [
                "L1_NCV",
                "L2_NCV"
            ]
        ]
        .mean(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QOM"] = (
        hourly[
            [
                "L1_QOM",
                "L2_QOM"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QGAS"] = (
        hourly[
            [
                "L1_QGAS",
                "L2_QGAS"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QNR"] = (
        hourly[
            [
                "L1_QNR",
                "L2_QNR"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_PFUM"] = (
        hourly[
            [
                "L1_PFUM",
                "L2_PFUM"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QAIR"] = (
        hourly[
            [
                "L1_QAIR",
                "L2_QAIR"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_W_OM_kg_s"] = (
        hourly[
            [
                "L1_W_OM_kg_s",
                "L2_W_OM_kg_s"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    # --------------------------------------------------------
    # 6) TARIH ALANLARI
    # --------------------------------------------------------

    print("\nGunluk, aylik ve yillik ortalamalar hesaplaniyor...")

    hourly["Date"] = (
        hourly["Time"]
        .dt.date
    )

    hourly["YearMonth"] = (
        hourly["Time"]
        .dt.to_period("M")
    )

    hourly["Year"] = (
        hourly["Time"]
        .dt.year
    )

    # --------------------------------------------------------
    # 7) NUMERIC KOLONLAR
    # --------------------------------------------------------

    numeric_cols = [
        c
        for c in hourly
        .select_dtypes(
            include=[np.number]
        )
        .columns
        if c not in [
            "valid_L1",
            "valid_L2"
        ]
    ]

    # --------------------------------------------------------
    # 8) DAILY
    # --------------------------------------------------------

    daily_avg = (
        hourly
        .groupby("Date")[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Date")
    )

    # --------------------------------------------------------
    # 9) MONTHLY
    # --------------------------------------------------------

    monthly_avg = (
        hourly
        .groupby("YearMonth")[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("YearMonth")
    )

    monthly_avg["YearMonth"] = (
        monthly_avg["YearMonth"]
        .astype(str)
    )

    # --------------------------------------------------------
    # 10) YEARLY
    # --------------------------------------------------------

    yearly_numeric_cols = [
        c
        for c in numeric_cols
        if c != "Year"
    ]

    yearly_avg = (
        hourly
        .groupby("Year")[yearly_numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Year")
    )

    # --------------------------------------------------------
    # 11) EXCEL WRITE
    # --------------------------------------------------------

    print("\nExcel yaziliyor:")
    print(OUTPUT_FILE)

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl"
    ) as writer:

        hourly.to_excel(
            writer,
            sheet_name="Hourly_Data",
            index=False
        )

        daily_avg.to_excel(
            writer,
            sheet_name="Daily_Averages",
            index=False
        )

        monthly_avg.to_excel(
            writer,
            sheet_name="Monthly_Averages",
            index=False
        )

        yearly_avg.to_excel(
            writer,
            sheet_name="Yearly_Averages",
            index=False
        )

    # --------------------------------------------------------
    # 12) SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("ISLEM TAMAMLANDI")
    print("=" * 70)

    print("\nSaatlik veriler:    Hourly_Data")
    print("Gunluk ortalamalar: Daily_Averages")
    print("Aylik ortalamalar:  Monthly_Averages")
    print("Yillik ortalamalar: Yearly_Averages")

    print(
        "\nW_OM hesaplarda 4 saatlik hareketli ortalama "
        "ile kullanildi."
    )

    print(
        f"NCV < {NCV_MIN:g} kJ/kg olan saatlerde "
        "NCV NaN kabul edildi."
    )

    print(
        f"4 saatlik waste flow < {MIN_WASTE_FLOW_TPH:g} t/h "
        "olan saatler invalid kabul edildi."
    )

    print("\nL1 valid saat:")
    print(
        int(
            hourly["valid_L1"]
            .fillna(False)
            .sum()
        )
    )

    print("\nL2 valid saat:")
    print(
        int(
            hourly["valid_L2"]
            .fillna(False)
            .sum()
        )
    )

    print("\nOutput dosyasi:")
    print(OUTPUT_FILE)


# ============================================================
#  CALISTIR
# ============================================================

main()

NCV CALCULATION

Calisma klasoru:
/workspaces/codespaces-jupyter

Input dosyasi:
/workspaces/codespaces-jupyter/data/TS hour raw value 2024 L1&2.xlsx

Excel okunuyor...


Excel okundu: 8,786 satir x 31 kolon

L1 hesaplaniyor...
L2 hesaplaniyor...

Durus/kalkis/trip saatleri filtreleniyor...

Gunluk, aylik ve yillik ortalamalar hesaplaniyor...

Excel yaziliyor:
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx

ISLEM TAMAMLANDI

Saatlik veriler:    Hourly_Data
Gunluk ortalamalar: Daily_Averages
Aylik ortalamalar:  Monthly_Averages
Yillik ortalamalar: Yearly_Averages

W_OM hesaplarda 4 saatlik hareketli ortalama ile kullanildi.
NCV < 3500 kJ/kg olan saatlerde NCV NaN kabul edildi.
4 saatlik waste flow < 15 t/h olan saatler invalid kabul edildi.

L1 valid saat:
7625

L2 valid saat:
7687

Output dosyasi:
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
#  DOSYA YOLLARI - GITHUB CODESPACES / JUPYTER
# ============================================================

BASE_DIR = Path.cwd()

INPUT_FILE = (
    BASE_DIR
    / "data"
    / "TS hour raw value 2024 L1&2.xlsx"
)

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "NCV_2024_hourly_L1_L2.xlsx"
)


# ============================================================
#  SABITLER
# ============================================================

NCV_GAS = 33500.0      # kJ/Nm3
TREF = 15.0            # C
VO2_MCR = 0.0764       # -
COLD_AIR_INJECTION = 5200.0 / 3600.0   # Nm3/s
H_EA = 551.0           # kJ/kg
H_WH = 3216.0          # kJ/kg
LOSS = 0.04            # %4 kayip

BASE_CONST = {
    "CPA": 1.3104,
    "CPF_TFUM": 1.3833,
    "CPF_TREF": 1.3574,
    "VWA": 3.94,
}

# NCV alt limiti (kJ/kg)
NCV_MIN = 3500.0

# 4 saatlik ortalama waste flow alt limiti (t/h)
MIN_WASTE_FLOW_TPH = 15.0


# ============================================================
#  YARDIMCI FONKSIYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    return x / 3600.0


def th_to_kgs(x):
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    return np.maximum(x, 0.0)


def get_cols(prefix: str) -> dict:
    """
    Verilen prefix icin ('1_' veya '2_') KKS tag isimlerini dondurur.
    """
    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2":  f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Gecersiz prefix: {prefix}")

    return base


def validate_columns(df: pd.DataFrame, cols: dict, label: str):
    missing = [c for c in cols.values() if c not in df.columns]

    if missing:
        print(f"\n{label} icin eksik kolonlar:")
        for c in missing:
            print(" -", c)

        raise KeyError(
            f"{label} hesabi yapilamadi. "
            f"{len(missing)} gerekli kolon Excel dosyasinda yok."
        )


# ============================================================
#  TEK HAT HESABI
# ============================================================

def compute_line(df: pd.DataFrame, prefix: str, label: str) -> pd.DataFrame:
    """
    Tek bir hat (L1 veya L2) icin NCV, QOM ve diger isi terimlerini hesaplar.

    W_OM hesaplamalarda 4 saatlik hareketli ortalama ile kullanilir.
    Eski kod mantigini korumak icin min_periods=1 kullanilir.

    NCV < 3500 kJ/kg olan saatlerde NCV NaN yapilir.
    """

    cols = get_cols(prefix)
    validate_columns(df, cols, label)

    def col(cname):
        return pd.to_numeric(df[cname], errors="coerce")

    # --------------------------------------------------------
    # AKISLAR
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(col(cols["W_AP"]))
    W_AS = nm3h_to_nm3s(col(cols["W_AS"]))

    W_GR = nm3h_to_nm3s(col(cols["W_GR"]))
    W_GL = nm3h_to_nm3s(col(cols["W_GL"]))

    W_WH_tph = col(cols["W_WH"])
    W_WH = th_to_kgs(W_WH_tph)

    # Ham waste flow t/h
    W_OM_tph_raw = col(cols["W_OM"])

    # 4 saatlik hareketli ortalama
    W_OM_tph_4h = (
        W_OM_tph_raw
        .rolling(window=4, min_periods=1)
        .mean()
    )

    # 4 saatlik ortalama kg/s
    W_OM = th_to_kgs(W_OM_tph_4h)

    # --------------------------------------------------------
    # SICAKLIKLAR
    # --------------------------------------------------------

    T_AP = col(cols["T_AP"])
    T_AS = col(cols["T_AS"])
    T_FUM = col(cols["TFUM"])

    # --------------------------------------------------------
    # GAZ ANALIZI
    # --------------------------------------------------------

    VO2 = pct_to_frac(col(cols["VO2"]))
    VH2O = pct_to_frac(col(cols["VH2O"]))

    # --------------------------------------------------------
    # KATSAYILAR
    # --------------------------------------------------------

    VWA = BASE_CONST["VWA"]
    CPF_TFUM = BASE_CONST["CPF_TFUM"]
    CPF_TREF = BASE_CONST["CPF_TREF"]
    CPA = BASE_CONST["CPA"]

    # --------------------------------------------------------
    # BACAGAZI
    # --------------------------------------------------------

    denominator = 1 - VH2O

    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    PFUM = (
        WFUM
        * (
            CPF_TFUM * T_FUM
            - CPF_TREF * TREF
        )
    )

    # --------------------------------------------------------
    # HAVA ISITMA
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        ) * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # BUHAR / SU
    # --------------------------------------------------------

    QNR = W_WH * (H_WH - H_EA)

    # --------------------------------------------------------
    # DOGAL GAZ
    # --------------------------------------------------------
    #
    # NOT:
    # Orijinal eski kodunda /1000 vardi.
    # Bu dosyada eski hesap mantigini korumak icin aynen birakildi.
    #

    QGAS = NCV_GAS * (W_GR + W_GL) / 1000.0

    # --------------------------------------------------------
    # ATIKTAN GELEN ISI
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV_raw = (
        QOM
        / W_OM.replace(0, np.nan)
    )

    # 3500 kJ/kg altini NaN yap
    NCV = NCV_raw.where(
        NCV_raw >= NCV_MIN,
        np.nan
    )

    # --------------------------------------------------------
    # SONUC
    # --------------------------------------------------------

    return pd.DataFrame({
        "Time": df["Time"],

        f"{label}_NCV": NCV,
        f"{label}_QOM": QOM,
        f"{label}_QGAS": QGAS,
        f"{label}_QNR": QNR,
        f"{label}_PFUM": PFUM,
        f"{label}_QAIR": QAIR,

        # 4 saatlik ortalama waste flow
        f"{label}_W_OM_kg_s": W_OM,

        # Ham saatlik waste flow
        f"{label}_W_OM_tph_raw": W_OM_tph_raw,

        # Bilgi amacli 4 saatlik ortalama t/h
        f"{label}_W_OM_tph_4h": W_OM_tph_4h,
    })


# ============================================================
#  ANA CALISMA
# ============================================================

def main():

    # --------------------------------------------------------
    # 1) INPUT KONTROL
    # --------------------------------------------------------

    print("=" * 70)
    print("NCV CALCULATION")
    print("=" * 70)

    print("\nCalisma klasoru:")
    print(BASE_DIR)

    print("\nInput dosyasi:")
    print(INPUT_FILE)

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            "\nExcel dosyasi bulunamadi.\n"
            f"Beklenen konum:\n{INPUT_FILE}\n\n"
            "Excel dosyasini repository icindeki data klasorune koy."
        )

    # --------------------------------------------------------
    # 2) VERI OKU
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        INPUT_FILE,
        engine="openpyxl"
    )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    # Ilk kolonu Time olarak kullan
    first_col = df.columns[0]

    if first_col != "Time":
        df = df.rename(
            columns={
                first_col: "Time"
            }
        )

    df["Time"] = pd.to_datetime(
        df["Time"],
        errors="coerce"
    )

    df = (
        df
        .sort_values("Time")
        .reset_index(drop=True)
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satir x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # 3) L1 / L2 HESAPLAMA
    # --------------------------------------------------------

    print("\nL1 hesaplaniyor...")
    L1 = compute_line(
        df,
        "1_",
        "L1"
    )

    print("L2 hesaplaniyor...")
    L2 = compute_line(
        df,
        "2_",
        "L2"
    )

    hourly = pd.merge(
        L1,
        L2,
        on="Time",
        how="outer"
    )

    # --------------------------------------------------------
    # 4) WASTE FLOW VALIDITY
    # --------------------------------------------------------

    print("\nDurus/kalkis/trip saatleri filtreleniyor...")

    min_waste_kg_s = th_to_kgs(
        MIN_WASTE_FLOW_TPH
    )

    # 4 saatlik ortalama W_OM >= 15 t/h ise valid
    hourly["valid_L1"] = (
        hourly["L1_W_OM_kg_s"]
        >= min_waste_kg_s
    )

    hourly["valid_L2"] = (
        hourly["L2_W_OM_kg_s"]
        >= min_waste_kg_s
    )

    # Geçersiz saatlerde hesap kolonlarini NaN yap
    # Waste flow bilgi kolonlarini koru

    protected_L1 = [
        "L1_W_OM_kg_s",
        "L1_W_OM_tph_raw",
        "L1_W_OM_tph_4h",
    ]

    protected_L2 = [
        "L2_W_OM_kg_s",
        "L2_W_OM_tph_raw",
        "L2_W_OM_tph_4h",
    ]

    cols_L1 = [
        c
        for c in hourly.columns
        if c.startswith("L1_")
        and c not in protected_L1
    ]

    cols_L2 = [
        c
        for c in hourly.columns
        if c.startswith("L2_")
        and c not in protected_L2
    ]

    hourly.loc[
        ~hourly["valid_L1"],
        cols_L1
    ] = np.nan

    hourly.loc[
        ~hourly["valid_L2"],
        cols_L2
    ] = np.nan

    # --------------------------------------------------------
    # 5) PLANT
    # --------------------------------------------------------
    #
    # Eski kod mantigi:
    # Plant NCV = L1/L2 NCV aritmetik ortalamasi
    #

    hourly["Plant_NCV"] = (
        hourly[
            [
                "L1_NCV",
                "L2_NCV"
            ]
        ]
        .mean(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QOM"] = (
        hourly[
            [
                "L1_QOM",
                "L2_QOM"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QGAS"] = (
        hourly[
            [
                "L1_QGAS",
                "L2_QGAS"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QNR"] = (
        hourly[
            [
                "L1_QNR",
                "L2_QNR"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_PFUM"] = (
        hourly[
            [
                "L1_PFUM",
                "L2_PFUM"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QAIR"] = (
        hourly[
            [
                "L1_QAIR",
                "L2_QAIR"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_W_OM_kg_s"] = (
        hourly[
            [
                "L1_W_OM_kg_s",
                "L2_W_OM_kg_s"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    # --------------------------------------------------------
    # 6) TARIH ALANLARI
    # --------------------------------------------------------

    print("\nGunluk, aylik ve yillik ortalamalar hesaplaniyor...")

    hourly["Date"] = (
        hourly["Time"]
        .dt.date
    )

    hourly["YearMonth"] = (
        hourly["Time"]
        .dt.to_period("M")
    )

    hourly["Year"] = (
        hourly["Time"]
        .dt.year
    )

    # --------------------------------------------------------
    # 7) NUMERIC KOLONLAR
    # --------------------------------------------------------

    numeric_cols = [
        c
        for c in hourly
        .select_dtypes(
            include=[np.number]
        )
        .columns
        if c not in [
            "valid_L1",
            "valid_L2"
        ]
    ]

    # --------------------------------------------------------
    # 8) DAILY
    # --------------------------------------------------------

    daily_avg = (
        hourly
        .groupby("Date")[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Date")
    )

    # --------------------------------------------------------
    # 9) MONTHLY
    # --------------------------------------------------------

    monthly_avg = (
        hourly
        .groupby("YearMonth")[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("YearMonth")
    )

    monthly_avg["YearMonth"] = (
        monthly_avg["YearMonth"]
        .astype(str)
    )

    # --------------------------------------------------------
    # 10) YEARLY
    # --------------------------------------------------------

    yearly_numeric_cols = [
        c
        for c in numeric_cols
        if c != "Year"
    ]

    yearly_avg = (
        hourly
        .groupby("Year")[yearly_numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Year")
    )

    # --------------------------------------------------------
    # 11) EXCEL WRITE
    # --------------------------------------------------------

    print("\nExcel yaziliyor:")
    print(OUTPUT_FILE)

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl"
    ) as writer:

        hourly.to_excel(
            writer,
            sheet_name="Hourly_Data",
            index=False
        )

        daily_avg.to_excel(
            writer,
            sheet_name="Daily_Averages",
            index=False
        )

        monthly_avg.to_excel(
            writer,
            sheet_name="Monthly_Averages",
            index=False
        )

        yearly_avg.to_excel(
            writer,
            sheet_name="Yearly_Averages",
            index=False
        )

    # --------------------------------------------------------
    # 12) SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("ISLEM TAMAMLANDI")
    print("=" * 70)

    print("\nSaatlik veriler:    Hourly_Data")
    print("Gunluk ortalamalar: Daily_Averages")
    print("Aylik ortalamalar:  Monthly_Averages")
    print("Yillik ortalamalar: Yearly_Averages")

    print(
        "\nW_OM hesaplarda 4 saatlik hareketli ortalama "
        "ile kullanildi."
    )

    print(
        f"NCV < {NCV_MIN:g} kJ/kg olan saatlerde "
        "NCV NaN kabul edildi."
    )

    print(
        f"4 saatlik waste flow < {MIN_WASTE_FLOW_TPH:g} t/h "
        "olan saatler invalid kabul edildi."
    )

    print("\nL1 valid saat:")
    print(
        int(
            hourly["valid_L1"]
            .fillna(False)
            .sum()
        )
    )

    print("\nL2 valid saat:")
    print(
        int(
            hourly["valid_L2"]
            .fillna(False)
            .sum()
        )
    )

    print("\nOutput dosyasi:")
    print(OUTPUT_FILE)


# ============================================================
#  CALISTIR
# ============================================================

main()

NCV CALCULATION

Calisma klasoru:
/workspaces/codespaces-jupyter

Input dosyasi:
/workspaces/codespaces-jupyter/data/TS hour raw value 2024 L1&2.xlsx

Excel okunuyor...


Excel okundu: 8,786 satir x 31 kolon

L1 hesaplaniyor...
L2 hesaplaniyor...

Durus/kalkis/trip saatleri filtreleniyor...

Gunluk, aylik ve yillik ortalamalar hesaplaniyor...

Excel yaziliyor:
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx

ISLEM TAMAMLANDI

Saatlik veriler:    Hourly_Data
Gunluk ortalamalar: Daily_Averages
Aylik ortalamalar:  Monthly_Averages
Yillik ortalamalar: Yearly_Averages

W_OM hesaplarda 4 saatlik hareketli ortalama ile kullanildi.
NCV < 3500 kJ/kg olan saatlerde NCV NaN kabul edildi.
4 saatlik waste flow < 15 t/h olan saatler invalid kabul edildi.

L1 valid saat:
7625

L2 valid saat:
7687

Output dosyasi:
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path


file = "TS hour raw value 2024 L1&2.xlsx"
df = pd.read_excel(file)

df.head()


# ============================================================
# SABİTLER
# ============================================================

CPA = 1.3104
CPF_TFUM = 1.087
CPF_TREF = 1.225
NCV_GAS = 33500.0
TREF = 15.0
VWA = 5.56
VO2_MCR = 0.0764
COLD_AIR_INJECTION = 5200.0 / 3600.0
H_EA = 551.0
H_WH = 3216.0
LOSS = 0.04


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    """Nm3/h -> Nm3/s"""
    return x / 3600.0


def th_to_kgs(x):
    """ton/h -> kg/s"""
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    """
    Yüzde değerlerini fraction'a çevirir.

    Örnek:
    8.5 -> 0.085
    0.085 -> 0.085
    """
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    """Negatif değerleri sıfıra çeker."""
    return np.maximum(x, 0)


# ============================================================
# KOLON EŞLEŞTİRME
# ============================================================

def get_cols(prefix):

    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2": f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    # Waste Flow kolon isimleri L1 ve L2 için farklı
    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Bilinmeyen line prefix: {prefix}")

    return base


# ============================================================
# KOLON KONTROLÜ
# ============================================================

def validate_columns(df, cols, line_name):

    missing = [
        column
        for column in cols.values()
        if column not in df.columns
    ]

    if missing:
        print(f"\n{line_name} için eksik kolonlar:")

        for column in missing:
            print(f"  - {column}")

        raise KeyError(
            f"{line_name} hesabı yapılamadı. "
            f"{len(missing)} adet gerekli kolon Excel dosyasında bulunamadı."
        )


# ============================================================
# ANA HESAPLAMA
# ============================================================

def compute_line(df, prefix):

    cols = get_cols(prefix)

    line_name = "L1" if prefix == "1_" else "L2"

    # Önce kolonları kontrol et
    validate_columns(df, cols, line_name)

    # Orijinal dataframe'i değiştirmemek için copy
    data = df.copy()

    # Kullanılacak kolonları numeric yap
    for column in cols.values():
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        )

    # --------------------------------------------------------
    # FLOW CONVERSIONS
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(data[cols["W_AP"]])
    W_AS = nm3h_to_nm3s(data[cols["W_AS"]])

    W_GR = nm3h_to_nm3s(data[cols["W_GR"]])
    W_GL = nm3h_to_nm3s(data[cols["W_GL"]])

    W_WH = th_to_kgs(data[cols["W_WH"]])
    W_OM = th_to_kgs(data[cols["W_OM"]])

    # --------------------------------------------------------
    # TEMPERATURES
    # --------------------------------------------------------

    T_AP = data[cols["T_AP"]]
    T_AS = data[cols["T_AS"]]
    T_FUM = data[cols["TFUM"]]

    # --------------------------------------------------------
    # FLUE GAS ANALYSIS
    # --------------------------------------------------------

    VO2 = pct_to_frac(data[cols["VO2"]])
    VH2O = pct_to_frac(data[cols["VH2O"]])

    # --------------------------------------------------------
    # FLUE GAS FLOW
    # --------------------------------------------------------

    denominator = 1 - VH2O

    # VH2O = 100% olması gibi matematiksel hataları önle
    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    # --------------------------------------------------------
    # FLUE GAS HEAT
    # --------------------------------------------------------

    PFUM = WFUM * (
        CPF_TFUM * T_FUM
        - CPF_TREF * TREF
    )

    # --------------------------------------------------------
    # AIR HEAT
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        )
        * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # WATER / STEAM ENERGY
    # --------------------------------------------------------

    QNR = W_WH * (
        H_WH - H_EA
    )

    # --------------------------------------------------------
    # GAS ENERGY
    # --------------------------------------------------------

    QGAS = NCV_GAS * (
        W_GR + W_GL
    )

    # --------------------------------------------------------
    # WASTE ENERGY
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV = QOM / W_OM.replace(
        0,
        np.nan
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    result = pd.DataFrame({
        "NCV": NCV,
        "QOM": QOM
    })

    return result


# ============================================================
# MAIN
# ============================================================

def main():

    # Bu Python dosyasının bulunduğu klasör
    BASE_DIR = Path.cwd()
    # Excel input
    input_file = (
        BASE_DIR
        / "data"
        / "TS hour raw value 2024 L1&2.xlsx"
    )

    # Output klasörü
    output_dir = BASE_DIR / "output"

    # Yoksa oluştur
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Output Excel
    output_file = (
        output_dir
        / "NCV_2024_hourly_L1_L2.xlsx"
    )

    # --------------------------------------------------------
    # INPUT FILE CONTROL
    # --------------------------------------------------------

    if not input_file.exists():

        raise FileNotFoundError(
            "\nInput Excel dosyası bulunamadı.\n"
            f"Beklenen konum:\n{input_file}\n\n"
            "Excel dosyasını repository içerisindeki "
            "'data' klasörüne koy."
        )

    print("=" * 60)
    print("NCV CALCULATION")
    print("=" * 60)

    print(f"\nInput:\n{input_file}")

    # --------------------------------------------------------
    # READ EXCEL
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        input_file,
        engine="openpyxl"
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satır x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # CALCULATIONS
    # --------------------------------------------------------

    print("\nL1 hesaplanıyor...")
    L1 = compute_line(df, "1_")

    print("L2 hesaplanıyor...")
    L2 = compute_line(df, "2_")

    # --------------------------------------------------------
    # SAVE EXCEL
    # --------------------------------------------------------

    print("\nSonuç Excel'e yazılıyor...")

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        L1.to_excel(
            writer,
            index=False,
            sheet_name="L1"
        )

        L2.to_excel(
            writer,
            index=False,
            sheet_name="L2"
        )

    print("\n" + "=" * 60)
    print("EXCEL OLUŞTURULDU")
    print("=" * 60)

    print(output_file)

    # --------------------------------------------------------
    # QUICK SUMMARY
    # --------------------------------------------------------

    print("\nL1 NCV Summary:")
    print(L1["NCV"].describe())

    print("\nL2 NCV Summary:")
    print(L2["NCV"].describe())


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

NCV CALCULATION

Input:
/workspaces/codespaces-jupyter/data/TS hour raw value 2024 L1&2.xlsx

Excel okunuyor...
Excel okundu: 8,786 satır x 31 kolon

L1 hesaplanıyor...
L2 hesaplanıyor...

Sonuç Excel'e yazılıyor...

EXCEL OLUŞTURULDU
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx

L1 NCV Summary:
count    7.255000e+03
mean     6.352186e+03
std      5.605623e+04
min     -4.622311e+06
25%      6.003715e+03
50%      6.736557e+03
75%      7.335058e+03
max      7.602992e+05
Name: NCV, dtype: float64

L2 NCV Summary:
count     7310.000000
mean      6797.563875
std       1397.380574
min     -13588.385354
25%       6181.667054
50%       6840.100764
75%       7431.839461
max      32031.949137
Name: NCV, dtype: float64


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
#  DOSYA YOLLARI - GITHUB CODESPACES / JUPYTER
# ============================================================

BASE_DIR = Path.cwd()

INPUT_FILE = (
    BASE_DIR
    / "data"
    / "TS hour raw value 2024 L1&2.xlsx"
)

OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "NCV_2024_hourly_L1_L2.xlsx"
)


# ============================================================
#  SABITLER
# ============================================================

NCV_GAS = 33500.0      # kJ/Nm3
TREF = 15.0            # C
VO2_MCR = 0.0764       # -
COLD_AIR_INJECTION = 5200.0 / 3600.0   # Nm3/s
H_EA = 551.0           # kJ/kg
H_WH = 3216.0          # kJ/kg
LOSS = 0.04            # %4 kayip

BASE_CONST = {
    "CPA": 1.3104,
    "CPF_TFUM": 1.3833,
    "CPF_TREF": 1.3574,
    "VWA": 3.94,
}

# NCV alt limiti (kJ/kg)
NCV_MIN = 3500.0

# 4 saatlik ortalama waste flow alt limiti (t/h)
MIN_WASTE_FLOW_TPH = 15.0


# ============================================================
#  YARDIMCI FONKSIYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    return x / 3600.0


def th_to_kgs(x):
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    return np.maximum(x, 0.0)


def get_cols(prefix: str) -> dict:
    """
    Verilen prefix icin ('1_' veya '2_') KKS tag isimlerini dondurur.
    """
    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2":  f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Gecersiz prefix: {prefix}")

    return base


def validate_columns(df: pd.DataFrame, cols: dict, label: str):
    missing = [c for c in cols.values() if c not in df.columns]

    if missing:
        print(f"\n{label} icin eksik kolonlar:")
        for c in missing:
            print(" -", c)

        raise KeyError(
            f"{label} hesabi yapilamadi. "
            f"{len(missing)} gerekli kolon Excel dosyasinda yok."
        )


# ============================================================
#  TEK HAT HESABI
# ============================================================

def compute_line(df: pd.DataFrame, prefix: str, label: str) -> pd.DataFrame:
    """
    Tek bir hat (L1 veya L2) icin NCV, QOM ve diger isi terimlerini hesaplar.

    W_OM hesaplamalarda 4 saatlik hareketli ortalama ile kullanilir.
    Eski kod mantigini korumak icin min_periods=1 kullanilir.

    NCV < 3500 kJ/kg olan saatlerde NCV NaN yapilir.
    """

    cols = get_cols(prefix)
    validate_columns(df, cols, label)

    def col(cname):
        return pd.to_numeric(df[cname], errors="coerce")

    # --------------------------------------------------------
    # AKISLAR
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(col(cols["W_AP"]))
    W_AS = nm3h_to_nm3s(col(cols["W_AS"]))

    W_GR = nm3h_to_nm3s(col(cols["W_GR"]))
    W_GL = nm3h_to_nm3s(col(cols["W_GL"]))

    W_WH_tph = col(cols["W_WH"])
    W_WH = th_to_kgs(W_WH_tph)

    # Ham waste flow t/h
    W_OM_tph_raw = col(cols["W_OM"])

    # 4 saatlik hareketli ortalama
    W_OM_tph_4h = (
        W_OM_tph_raw
        .rolling(window=4, min_periods=1)
        .mean()
    )

    # 4 saatlik ortalama kg/s
    W_OM = th_to_kgs(W_OM_tph_4h)

    # --------------------------------------------------------
    # SICAKLIKLAR
    # --------------------------------------------------------

    T_AP = col(cols["T_AP"])
    T_AS = col(cols["T_AS"])
    T_FUM = col(cols["TFUM"])

    # --------------------------------------------------------
    # GAZ ANALIZI
    # --------------------------------------------------------

    VO2 = pct_to_frac(col(cols["VO2"]))
    VH2O = pct_to_frac(col(cols["VH2O"]))

    # --------------------------------------------------------
    # KATSAYILAR
    # --------------------------------------------------------

    VWA = BASE_CONST["VWA"]
    CPF_TFUM = BASE_CONST["CPF_TFUM"]
    CPF_TREF = BASE_CONST["CPF_TREF"]
    CPA = BASE_CONST["CPA"]

    # --------------------------------------------------------
    # BACAGAZI
    # --------------------------------------------------------

    denominator = 1 - VH2O

    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    PFUM = (
        WFUM
        * (
            CPF_TFUM * T_FUM
            - CPF_TREF * TREF
        )
    )

    # --------------------------------------------------------
    # HAVA ISITMA
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        ) * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # BUHAR / SU
    # --------------------------------------------------------

    QNR = W_WH * (H_WH - H_EA)

    # --------------------------------------------------------
    # DOGAL GAZ
    # --------------------------------------------------------
    #
    # NOT:
    # Orijinal eski kodunda /1000 vardi.
    # Bu dosyada eski hesap mantigini korumak icin aynen birakildi.
    #

    QGAS = NCV_GAS * (W_GR + W_GL) / 1000.0

    # --------------------------------------------------------
    # ATIKTAN GELEN ISI
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV_raw = (
        QOM
        / W_OM.replace(0, np.nan)
    )

    # 3500 kJ/kg altini NaN yap
    NCV = NCV_raw.where(
        NCV_raw >= NCV_MIN,
        np.nan
    )

    # --------------------------------------------------------
    # SONUC
    # --------------------------------------------------------

    return pd.DataFrame({
        "Time": df["Time"],

        f"{label}_NCV": NCV,
        f"{label}_QOM": QOM,
        f"{label}_QGAS": QGAS,
        f"{label}_QNR": QNR,
        f"{label}_PFUM": PFUM,
        f"{label}_QAIR": QAIR,

        # 4 saatlik ortalama waste flow
        f"{label}_W_OM_kg_s": W_OM,

        # Ham saatlik waste flow
        f"{label}_W_OM_tph_raw": W_OM_tph_raw,

        # Bilgi amacli 4 saatlik ortalama t/h
        f"{label}_W_OM_tph_4h": W_OM_tph_4h,
    })


# ============================================================
#  ANA CALISMA
# ============================================================

def main():

    # --------------------------------------------------------
    # 1) INPUT KONTROL
    # --------------------------------------------------------

    print("=" * 70)
    print("NCV CALCULATION")
    print("=" * 70)

    print("\nCalisma klasoru:")
    print(BASE_DIR)

    print("\nInput dosyasi:")
    print(INPUT_FILE)

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            "\nExcel dosyasi bulunamadi.\n"
            f"Beklenen konum:\n{INPUT_FILE}\n\n"
            "Excel dosyasini repository icindeki data klasorune koy."
        )

    # --------------------------------------------------------
    # 2) VERI OKU
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        INPUT_FILE,
        engine="openpyxl"
    )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    # Ilk kolonu Time olarak kullan
    first_col = df.columns[0]

    if first_col != "Time":
        df = df.rename(
            columns={
                first_col: "Time"
            }
        )

    df["Time"] = pd.to_datetime(
        df["Time"],
        errors="coerce"
    )

    df = (
        df
        .sort_values("Time")
        .reset_index(drop=True)
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satir x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # 3) L1 / L2 HESAPLAMA
    # --------------------------------------------------------

    print("\nL1 hesaplaniyor...")
    L1 = compute_line(
        df,
        "1_",
        "L1"
    )

    print("L2 hesaplaniyor...")
    L2 = compute_line(
        df,
        "2_",
        "L2"
    )

    hourly = pd.merge(
        L1,
        L2,
        on="Time",
        how="outer"
    )

    # --------------------------------------------------------
    # 4) WASTE FLOW VALIDITY
    # --------------------------------------------------------

    print("\nDurus/kalkis/trip saatleri filtreleniyor...")

    min_waste_kg_s = th_to_kgs(
        MIN_WASTE_FLOW_TPH
    )

    # 4 saatlik ortalama W_OM >= 15 t/h ise valid
    hourly["valid_L1"] = (
        hourly["L1_W_OM_kg_s"]
        >= min_waste_kg_s
    )

    hourly["valid_L2"] = (
        hourly["L2_W_OM_kg_s"]
        >= min_waste_kg_s
    )

    # Geçersiz saatlerde hesap kolonlarini NaN yap
    # Waste flow bilgi kolonlarini koru

    protected_L1 = [
        "L1_W_OM_kg_s",
        "L1_W_OM_tph_raw",
        "L1_W_OM_tph_4h",
    ]

    protected_L2 = [
        "L2_W_OM_kg_s",
        "L2_W_OM_tph_raw",
        "L2_W_OM_tph_4h",
    ]

    cols_L1 = [
        c
        for c in hourly.columns
        if c.startswith("L1_")
        and c not in protected_L1
    ]

    cols_L2 = [
        c
        for c in hourly.columns
        if c.startswith("L2_")
        and c not in protected_L2
    ]

    hourly.loc[
        ~hourly["valid_L1"],
        cols_L1
    ] = np.nan

    hourly.loc[
        ~hourly["valid_L2"],
        cols_L2
    ] = np.nan

    # --------------------------------------------------------
    # 5) PLANT
    # --------------------------------------------------------
    #
    # Eski kod mantigi:
    # Plant NCV = L1/L2 NCV aritmetik ortalamasi
    #

    hourly["Plant_NCV"] = (
        hourly[
            [
                "L1_NCV",
                "L2_NCV"
            ]
        ]
        .mean(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QOM"] = (
        hourly[
            [
                "L1_QOM",
                "L2_QOM"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QGAS"] = (
        hourly[
            [
                "L1_QGAS",
                "L2_QGAS"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QNR"] = (
        hourly[
            [
                "L1_QNR",
                "L2_QNR"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_PFUM"] = (
        hourly[
            [
                "L1_PFUM",
                "L2_PFUM"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_QAIR"] = (
        hourly[
            [
                "L1_QAIR",
                "L2_QAIR"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    hourly["Plant_W_OM_kg_s"] = (
        hourly[
            [
                "L1_W_OM_kg_s",
                "L2_W_OM_kg_s"
            ]
        ]
        .sum(
            axis=1,
            skipna=True
        )
    )

    # --------------------------------------------------------
    # 6) TARIH ALANLARI
    # --------------------------------------------------------

    print("\nGunluk, aylik ve yillik ortalamalar hesaplaniyor...")

    hourly["Date"] = (
        hourly["Time"]
        .dt.date
    )

    hourly["YearMonth"] = (
        hourly["Time"]
        .dt.to_period("M")
    )

    hourly["Year"] = (
        hourly["Time"]
        .dt.year
    )

    # --------------------------------------------------------
    # 7) NUMERIC KOLONLAR
    # --------------------------------------------------------

    numeric_cols = [
        c
        for c in hourly
        .select_dtypes(
            include=[np.number]
        )
        .columns
        if c not in [
            "valid_L1",
            "valid_L2"
        ]
    ]

    # --------------------------------------------------------
    # 8) DAILY
    # --------------------------------------------------------

    daily_avg = (
        hourly
        .groupby("Date")[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Date")
    )

    # --------------------------------------------------------
    # 9) MONTHLY
    # --------------------------------------------------------

    monthly_avg = (
        hourly
        .groupby("YearMonth")[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("YearMonth")
    )

    monthly_avg["YearMonth"] = (
        monthly_avg["YearMonth"]
        .astype(str)
    )

    # --------------------------------------------------------
    # 10) YEARLY
    # --------------------------------------------------------

    yearly_numeric_cols = [
        c
        for c in numeric_cols
        if c != "Year"
    ]

    yearly_avg = (
        hourly
        .groupby("Year")[yearly_numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Year")
    )

    # --------------------------------------------------------
    # 11) EXCEL WRITE
    # --------------------------------------------------------

    print("\nExcel yaziliyor:")
    print(OUTPUT_FILE)

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl"
    ) as writer:

        hourly.to_excel(
            writer,
            sheet_name="Hourly_Data",
            index=False
        )

        daily_avg.to_excel(
            writer,
            sheet_name="Daily_Averages",
            index=False
        )

        monthly_avg.to_excel(
            writer,
            sheet_name="Monthly_Averages",
            index=False
        )

        yearly_avg.to_excel(
            writer,
            sheet_name="Yearly_Averages",
            index=False
        )

    # --------------------------------------------------------
    # 12) SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("ISLEM TAMAMLANDI")
    print("=" * 70)

    print("\nSaatlik veriler:    Hourly_Data")
    print("Gunluk ortalamalar: Daily_Averages")
    print("Aylik ortalamalar:  Monthly_Averages")
    print("Yillik ortalamalar: Yearly_Averages")

    print(
        "\nW_OM hesaplarda 4 saatlik hareketli ortalama "
        "ile kullanildi."
    )

    print(
        f"NCV < {NCV_MIN:g} kJ/kg olan saatlerde "
        "NCV NaN kabul edildi."
    )

    print(
        f"4 saatlik waste flow < {MIN_WASTE_FLOW_TPH:g} t/h "
        "olan saatler invalid kabul edildi."
    )

    print("\nL1 valid saat:")
    print(
        int(
            hourly["valid_L1"]
            .fillna(False)
            .sum()
        )
    )

    print("\nL2 valid saat:")
    print(
        int(
            hourly["valid_L2"]
            .fillna(False)
            .sum()
        )
    )

    print("\nOutput dosyasi:")
    print(OUTPUT_FILE)


# ============================================================
#  CALISTIR
# ============================================================

main()

NCV CALCULATION

Calisma klasoru:
/workspaces/codespaces-jupyter

Input dosyasi:
/workspaces/codespaces-jupyter/data/TS hour raw value 2024 L1&2.xlsx

Excel okunuyor...
Excel okundu: 8,786 satir x 31 kolon

L1 hesaplaniyor...
L2 hesaplaniyor...

Durus/kalkis/trip saatleri filtreleniyor...

Gunluk, aylik ve yillik ortalamalar hesaplaniyor...

Excel yaziliyor:
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx

ISLEM TAMAMLANDI

Saatlik veriler:    Hourly_Data
Gunluk ortalamalar: Daily_Averages
Aylik ortalamalar:  Monthly_Averages
Yillik ortalamalar: Yearly_Averages

W_OM hesaplarda 4 saatlik hareketli ortalama ile kullanildi.
NCV < 3500 kJ/kg olan saatlerde NCV NaN kabul edildi.
4 saatlik waste flow < 15 t/h olan saatler invalid kabul edildi.

L1 valid saat:
7625

L2 valid saat:
7687

Output dosyasi:
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx
